# Data preprocessing Notebook —  Character RAG Bot

Main Steps
1. Load configuration from config.yaml
2. Select target character, such as rafayel
3. Load all .txt script files from the target character folder
4. Parse dialogue lines by speaker, such as Rafayel: and Main Character:
5. Create dialogue context by combining Rafayel’s dialogue with previous and next dialogue turns
6. Save processed dialogue data into the dialogues/ folder

In [53]:
root = '..'
data_folder = 'data'
config_file = 'config.yaml'

In [54]:
from pathlib import Path
import os
import re
from tqdm.notebook import tqdm

## Load config

In [55]:
from pathlib import Path
import yaml

def load_config(config_path: str = "config.yaml"):
    root = Path.cwd()
    config_file = root / config_path

    if not config_file.exists():
        raise FileNotFoundError(f"Config file not found: {config_file}")

    with open(config_file, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    return config


def resolve_path(path_value: str) -> Path:
    root = Path.cwd()
    path = Path(path_value)

    if path.is_absolute():
        return path

    return root / path

In [56]:
config = load_config()

list_of_characters = config["LIST_OF_CHARACTER"]
target_character = config["TARGET_CHARACTER"]

character_synonyms = config["CHARACTER_SYNONYMS"]
data_list_of_character = config["DATA_LIST_OF_CHARACTER"]

dialogue_folder = resolve_path(config["DIALOGUE_FOLDER"])
max_context_length = config["MAX_CONTEXT_LENGTH"]
dialogues_joiner = config["DIALOGUES_JOINER"]

print("Target:", target_character)
print("Characters:", list_of_characters)
print("Synonyms:", character_synonyms[target_character])
print("Data folders:", data_list_of_character[target_character])

Target: rafayel
Characters: ['xavier', 'zayne', 'rafayel', 'sylus', 'caleb', 'mc']
Synonyms: ['rafayel', 'ราฟาเอล', 'ราฟ', 'raf', 'rafael']
Data folders: ['./data/rafayel_scripts']


In [57]:
target_character = config["TARGET_CHARACTER"]
script_folders = config["DATA_LIST_OF_CHARACTER"][target_character]
script_files = []

for folder in script_folders:
    folder_path = resolve_path(folder)
    script_files.extend(folder_path.rglob("*.txt"))
print(f"Found {len(script_files)} script files for {target_character}")

Found 31 script files for rafayel


In [58]:
def extract_text_from_txt(txt_path: Path) -> str:
    """
    Read text from a .txt script file.
    """
    return txt_path.read_text(encoding="utf-8")

In [59]:
def get_all_character_names(character: str, character_synonyms: dict) -> list[str]:
    """
    Get all possible names/aliases for a character.
    Example:
    rafayel -> Rafayel, rafayel, ราฟาเอล, ราฟ, raf, rafael
    """
    names = character_synonyms.get(character, [character])

    all_names = []

    for name in names:
        if not name:
            continue

        name = str(name).strip()

        all_names.append(name)
        all_names.append(name.lower())
        all_names.append(name.upper())
        all_names.append(name.title())

    return list(dict.fromkeys(all_names))

In [60]:
def build_alias_to_character_map(character_synonyms: dict) -> dict:
    alias_map = {}

    display_names = {
        "xavier": "Xavier",
        "zayne": "Zayne",
        "rafayel": "Rafayel",
        "sylus": "Sylus",
        "caleb": "Caleb",
        "mc": "Main Character",
    }

    for character, aliases in character_synonyms.items():
        display_name = display_names.get(character, character)

        for alias in aliases:
            alias = str(alias).strip()
            if not alias:
                continue

            alias_map[alias.lower()] = display_name

    return alias_map

In [61]:
def parse_dialogues(script_text: str, character_synonyms: dict) -> list[dict]:
    """
    Parse dialogue lines like:
    Rafayel: ...
    Main Character: ...
    """

    alias_map = build_alias_to_character_map(character_synonyms)

    # sort by length so "Main Character" matches before "MC"
    aliases = sorted(alias_map.keys(), key=len, reverse=True)

    speaker_pattern = "|".join(re.escape(alias) for alias in aliases)

    pattern = re.compile(
        rf"^\s*({speaker_pattern})\s*:\s*(.+?)\s*$",
        re.IGNORECASE | re.MULTILINE
    )

    dialogues = []

    for match in pattern.finditer(script_text):
        raw_speaker = match.group(1).strip()
        text = match.group(2).strip()

        speaker = alias_map.get(raw_speaker.lower(), raw_speaker)

        dialogues.append({
            "speaker": speaker,
            "text": text,
            "start": match.start(),
            "end": match.end(),
        })

    return dialogues

In [62]:
sample_text = extract_text_from_txt(script_files[0])
dialogues = parse_dialogues(sample_text, character_synonyms)

print("dialogues:", len(dialogues))

for d in dialogues[:10]:
    print(d["speaker"], ":", d["text"])

dialogues: 447
Main Character : Sure thing, but I can’t promise I’l succeed. Is that okay with you?
Main Character : Sigh… He ran away already…
Rafayel : Unfortunate. This species of fish can only survive for a week on land.
Main Character : (…A tourist?)
Rafayel : The fish is gonna slip away, you know.
Rafayel : Ta-da.
Rafayel : The owner probably just wanted to throw in some fish to fit the theme.
Rafayel : But this one, bright as a flame, is a real Flammula from Lemurian legends.
Main Character : Flammula? I’m not very familiar with those myths or folklore…
Main Character : Oh.


In [63]:
def combine_dialogue_with_context(
    dialogues: list[dict],
    target_speaker: str = "Rafayel",
    context_turns: int = 2
) -> list[str]:
    """
    Combine each target speaker dialogue with previous and next dialogue turns.
    """

    dialogue_with_context_all = []

    for index, dialogue in enumerate(dialogues):
        if dialogue["speaker"] != target_speaker:
            continue

        start = max(0, index - context_turns)
        end = min(len(dialogues), index + context_turns + 1)

        before_dialogues = dialogues[start:index]
        after_dialogues = dialogues[index + 1:end]

        parts = []

        if before_dialogues:
            before_text = "\n".join(
                f"{item['speaker']}: {item['text']}"
                for item in before_dialogues
            )
            parts.append(f"Previous Dialogue:\n{before_text}")

        parts.append(
            f"{target_speaker} Dialogue:\n"
            f"{target_speaker}: {dialogue['text']}"
        )

        if after_dialogues:
            after_text = "\n".join(
                f"{item['speaker']}: {item['text']}"
                for item in after_dialogues
            )
            parts.append(f"Next Dialogue:\n{after_text}")

        content = "\n\n".join(parts)
        dialogue_with_context_all.append(content)

    return dialogue_with_context_all

In [64]:
target_speaker = "Rafayel"

dialogue_save_path = dialogue_folder / target_character
dialogue_save_path.mkdir(parents=True, exist_ok=True)

for script_file in script_files:
    script_text = extract_text_from_txt(script_file)

    dialogues = parse_dialogues(
        script_text=script_text,
        character_synonyms=character_synonyms
    )

    dialogues_with_context = combine_dialogue_with_context(
        dialogues=dialogues,
        target_speaker=target_speaker,
        context_turns=2
    )

    dialogues_with_context_combined = dialogues_joiner.join(dialogues_with_context)

    save_script_name = script_file.stem + "_dialogues.txt"
    save_path = dialogue_save_path / save_script_name

    save_path.write_text(dialogues_with_context_combined, encoding="utf-8")

    print(f"Saved: {save_path} | dialogues: {len(dialogues_with_context)}")

Saved: D:\RAG\dialogues\rafayel\01_mainstory_dialogues.txt | dialogues: 236
Saved: D:\RAG\dialogues\rafayel\01_message_dialogues.txt | dialogues: 10
Saved: D:\RAG\dialogues\rafayel\01_tender_dialogues.txt | dialogues: 68
Saved: D:\RAG\dialogues\rafayel\02_mainstory_dialogues.txt | dialogues: 228
Saved: D:\RAG\dialogues\rafayel\02_message_dialogues.txt | dialogues: 6
Saved: D:\RAG\dialogues\rafayel\02_tender_dialogues.txt | dialogues: 63
Saved: D:\RAG\dialogues\rafayel\03_mainstory_dialogues.txt | dialogues: 60
Saved: D:\RAG\dialogues\rafayel\03_message_dialogues.txt | dialogues: 8
Saved: D:\RAG\dialogues\rafayel\03_tender_dialogues.txt | dialogues: 54
Saved: D:\RAG\dialogues\rafayel\04_mainstory_dialogues.txt | dialogues: 163
Saved: D:\RAG\dialogues\rafayel\04_message_dialogues.txt | dialogues: 11
Saved: D:\RAG\dialogues\rafayel\04_tender_dialogues.txt | dialogues: 71
Saved: D:\RAG\dialogues\rafayel\05_tender_dialogues.txt | dialogues: 65
Saved: D:\RAG\dialogues\rafayel\06_tender_dialo